In [1]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
print("Working ✅")

Working ✅


In [2]:
df=pd.read_csv("../src/data/raw/bank-additional-full.csv",sep=";")
print(df.columns)

Index(['age', 'job', 'marital', 'education', 'default', 'housing', 'loan',
       'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx',
       'cons.conf.idx', 'euribor3m', 'nr.employed', 'y'],
      dtype='object')


In [3]:
import os
print(os.getcwd())

c:\Users\Shravani\Desktop\Projects\mlops-drift-monitor\notebooks


In [4]:
X=df.drop("y",axis=1)
#convert target variable to binary
y=df["y"].str.strip().map({"yes":1,"no":0})

In [5]:
print(X.shape)
print(y.shape)
print(y.head())
print(y.unique())

(41188, 20)
(41188,)
0    0
1    0
2    0
3    0
4    0
Name: y, dtype: int64
[0 1]


In [6]:
#Give me all columns that are numeric
num_cols=X.select_dtypes(include=["int64","float64"]).columns
print(num_cols)

#Give me all columns that are categorical
cat_cols=X.select_dtypes(include=["object"]).columns
print(cat_cols)

Index(['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate',
       'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed'],
      dtype='object')
Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
       'month', 'day_of_week', 'poutcome'],
      dtype='object')


In [7]:
model = GradientBoostingClassifier()
model_name = "GradientBoostingClassifier"

#We will now build a preprocessor
#Here ColumnTransformer allows us to apply different transformations to different columns
#handle_unknown="ignore" is used to ignore any unknown categories during transformation
#Pipeline allows us to chain multiple steps together, in this case preprocessing and modeling

preprocessor=ColumnTransformer([
    ("num",StandardScaler(),num_cols),
    ("cat",OneHotEncoder(handle_unknown="ignore"),cat_cols)
])

pipeline=Pipeline([
    ("preprocessing",preprocessor),
    ("model",model)
])

In [8]:
#Split the data into train and test sets
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [9]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_registry_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Default")

with mlflow.start_run():
    
   #Fit the pipeline on the training data
    pipeline.fit(X_train,y_train)

    #Predict on the test data
    preds=pipeline.predict(X_test)

    #Evaluate the model
    acc=accuracy_score(y_test,preds)
    print(f"Accuracy: {acc*100:.2f}%")
    
    #Log the model and metrics to MLflow
    #means that we are logging the accuracy metric with the name "accuracy" and the value of acc
    mlflow.log_metric("accuracy",acc)
    
    #means that we are logging a parameter with the name "model" and the value "GradientBoostingClassifier"
    mlflow.log_param("model",model_name)
    
    #Log the model to MLflow
    #means that we are logging the pipeline object as a model with the name "model"
    mlflow.sklearn.log_model(
       sk_model=pipeline,
       artifact_path="model",
       registered_model_name="bank-marketing-model")
    
    

Accuracy: 91.89%


2026/04/04 12:05:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/04 12:05:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'bank-marketing-model' already exists. Creating a new version of this model...
2026/04/04 12:06:02 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: bank-marketing-model, version 5


🏃 View run nosy-midge-282 at: http://127.0.0.1:5000/#/experiments/0/runs/740a24a793824aacb75900ba11737887
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0


Created version '5' of model 'bank-marketing-model'.


In [10]:
from scipy.stats import ks_2samp
drift_res=[]
prod_df=df.copy()
prod_df=prod_df[prod_df["age"]<35]

train_age=df["age"] 
prod_age=prod_df["age"] 

print(train_age.mean(),prod_age.mean())

for col in num_cols:
    train_col=df[col]
    prod_col=prod_df[col]
    ks_stat, p_value = ks_2samp(train_col, prod_col)
    drift_res.append((col, ks_stat, p_value))
    
    
for r in drift_res:
    print(f"Column: {r[0]}, KS Statistic: {r[1]:.4f}, P-value: {r[2]:.4f}")

40.02406040594348 29.89433374000271
Column: age, KS Statistic: 0.6418, P-value: 0.0000
Column: duration, KS Statistic: 0.0049, P-value: 0.9578
Column: campaign, KS Statistic: 0.0067, P-value: 0.7058
Column: pdays, KS Statistic: 0.0049, P-value: 0.9587
Column: previous, KS Statistic: 0.0123, P-value: 0.0726
Column: emp.var.rate, KS Statistic: 0.0598, P-value: 0.0000
Column: cons.price.idx, KS Statistic: 0.0500, P-value: 0.0000
Column: cons.conf.idx, KS Statistic: 0.0601, P-value: 0.0000
Column: euribor3m, KS Statistic: 0.0614, P-value: 0.0000
Column: nr.employed, KS Statistic: 0.0598, P-value: 0.0000


In [11]:
import pandas as pd
drift_df=pd.DataFrame(drift_res,columns=["feature","ks_stat","p_value"])
drift_df["drift"]=drift_df["p_value"]<0.05

print(drift_df)

          feature   ks_stat       p_value  drift
0             age  0.641789  0.000000e+00   True
1        duration  0.004869  9.578254e-01  False
2        campaign  0.006732  7.058326e-01  False
3           pdays  0.004858  9.586876e-01  False
4        previous  0.012340  7.255285e-02  False
5    emp.var.rate  0.059771  3.534025e-34   True
6  cons.price.idx  0.050018  4.657904e-24   True
7   cons.conf.idx  0.060111  1.456893e-34   True
8       euribor3m  0.061398  4.835628e-36   True
9     nr.employed  0.059771  3.534025e-34   True


In [12]:
drifted_features=drift_df[drift_df["drift"]==True]["feature"].tolist()
print("Drifted features:",drifted_features)

Drifted features: ['age', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']


In [13]:
if "age" in drifted_features:
    print("Age feature has drifted likely caused by changes in the data distribution. Consider retraining the model with updated data.")

Age feature has drifted likely caused by changes in the data distribution. Consider retraining the model with updated data.


In [19]:
prompt = f"""
You are an ML monitoring expert.

Drift detected in features: {drifted_features}

Key stats:
- Age mean changed from {train_age.mean():.2f} to {prod_age.mean():.2f}

CONTEXT:
This drift was artificially created by filtering dataset to only include users with age < 35.

INSTRUCTIONS:
- Identify the EXACT cause of drift based on this context
- DO NOT give generic reasons like economy, population, etc.
- Keep answer short (max 4-5 lines)
- Suggest 2 practical ML actions

Answer:
"""

print(prompt)


You are an ML monitoring expert.

Drift detected in features: ['age', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

Key stats:
- Age mean changed from 40.02 to 29.89

CONTEXT:
This drift was artificially created by filtering dataset to only include users with age < 35.

INSTRUCTIONS:
- Identify the EXACT cause of drift based on this context
- DO NOT give generic reasons like economy, population, etc.
- Keep answer short (max 4-5 lines)
- Suggest 2 practical ML actions

Answer:



In [20]:
import ollama
def llm_explain_drift(prompt):
    response=ollama.chat(model='phi3:mini',
                         messages=[
                             {'role':"user", "content":prompt}
                         ])
    return response['message']['content']

In [21]:
llm_output=llm_explain_drift(prompt)
print(llm_output)

The exact cause for feature distribution changes was the filtering dataset to only include users with age < 35 years old which skewed younger than before, hence altering mean ages and potentially related features. Two practices in machine learning that can counteract this drift are online learning where models continuously learn from new data patterns or employ retraining strategies at regular intervals using the latest representative dataset samples to maintain model accuracy over time.
